# Engine-Level Multi-Instance TCN-BiGRU + XGBoost for CMAPSS

This notebook replaces window-level test decisions with engine-level multi-instance aggregation.

Instead of treating every sliding window as an independent test sample, each engine snapshot is represented as a bag of recent windows. The trained TCN-BiGRU-Attention encoder embeds the windows, bag-level statistics aggregate the embeddings, and XGBoost predicts whether the engine snapshot is near failure.

Default target is FD004 because that is where sliding window evaluation was weakest.

In [5]:
# Run only if the environment is missing dependencies.
INSTALL_MISSING_PACKAGES = False

if INSTALL_MISSING_PACKAGES:
    import subprocess
    import sys

    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'xgboost', 'joblib'])


In [6]:
import copy
import pandas as pd

from engine_level_mil_tcn_bigru_xgboost import DEFAULT_CFG, run_engine_mil_pipeline

cfg = copy.deepcopy(DEFAULT_CFG)
cfg.update({
    'dataset': 'FD004',
    'data_dir': 'data',
    'rul_threshold': 30,
    'seq_len': 40,
    'snapshot_stride': 10,
    'bag_window_stride': 2,
    'max_windows_per_bag': 60,
    'recent_windows_only': True,
})

cfg


ModuleNotFoundError: No module named 'engine_level_mil_tcn_bigru_xgboost'

## Run FD004

This is the main replacement for the weak FD004 sliding-window result. It evaluates one engine-level prediction per test engine.

In [ ]:
metrics, artifacts = run_engine_mil_pipeline('FD004', cfg)
pd.DataFrame([metrics]).drop(columns=['Best Params'])


## Optional All-Dataset Run

Use this only after FD004 is working. It trains a separate encoder and XGBoost model for each dataset.

In [ ]:
RUN_ALL_DATASETS = False

if RUN_ALL_DATASETS:
    rows = []
    for fd in ['FD001', 'FD002', 'FD003', 'FD004']:
        fd_cfg = copy.deepcopy(cfg)
        fd_cfg['dataset'] = fd
        result, _ = run_engine_mil_pipeline(fd, fd_cfg)
        rows.append(result)
    results = pd.DataFrame(rows)
    display(results.drop(columns=['Best Params']))
else:
    print('Set RUN_ALL_DATASETS = True to run FD001-FD004.')
